# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Houssem-Bjaoui/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Lane 2 — Refresh / Content Opportunity Scoring**

The lane's core question is: *Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?* My initial focus within this lane is the **refresh / opportunity prioritization** slice of that question — specifically, which existing pages look worth a human content review first, based on signals that were observable before any decision was made about them.

**Why this lane, and why it's more than "train a model":**

SEO and content teams almost always have more pages than review time. Someone has to decide, every week, which pages get looked at first. Right now that decision is often made informally — gut feeling, whichever page someone happens to notice, or a simple rule of thumb. A ranked, evidence-backed queue could make that triage faster and more consistent.

This is a decision-support problem, not just a modeling exercise, because:

- The end product is a **ranked list a human acts on**, not a model score sitting in a notebook. The model (once built) is one ingredient in a workflow that starts with a business question (which pages matter?) and ends with a human decision (review this page or not).
- A model only becomes useful here if it **beats a simple, transparent baseline rule** (e.g., "stale and still getting impressions") — otherwise the team is better off with the simple rule they can already explain to a client.
- Every prediction needs a **reason code** a reviewer can inspect (e.g., "declining with demand", "thin visible page") — a black-box score without a reason is not something a content team can act on responsibly.
- The lane guide is explicit that this data can show **association and observed patterns**, not proof that a refresh will fix anything. So part of the work is being honest about what the output can and can't claim — that discipline matters as much as the score itself.

In short: the interesting problem isn't "can I train a classifier," it's "can I turn messy, partial, observational signals into a prioritized list that a real reviewer would trust enough to act on, and can I say honestly where that list might be wrong."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

**Provisional research question:**

> Which pages should an SEO team prioritize for content refresh first, based on observable performance, freshness, demand, and content signals?

**The decision this supports:** Which pages should be reviewed first, out of a much larger set the team does not have time to check individually? This is a triage/ranking decision, not a yes/no decision about any single page in isolation.

**The action someone could take:** An SEO or content team member reviews the highest-priority pages first and decides — for each one — whether to refresh the content, expand it, protect it as-is, prune it, or simply keep monitoring it. The model or score does not take the action; it points a human at where to look first.

**The cost of a wrong recommendation:**

- **False positive** (page flagged as a priority, but it didn't actually need attention): the team spends limited review or writing time on a page that had nothing meaningfully wrong with it, and a page that genuinely needed help waits longer.
- **False negative** (a page with a real opportunity is never flagged): a page that is quietly losing visibility, or that has real untapped demand, goes unreviewed and the opportunity is missed — possibly for months, until someone notices some other way.

Because both directions of error have a real cost, the goal is not just "maximize accuracy" — it's to build a ranked queue where the *top* of the list is trustworthy enough that a reviewer's limited time is well spent, which is why later sections will care about metrics like precision at the top of the list (precision@K), not just an overall accuracy number.

**What this research question is *not* claiming:**

- It does not claim that refreshing a flagged page will definitely improve its performance — only that the page looks *worth a human look*, based on patterns observed in the data.
- It does not claim that any single signal (position, freshness, word count, etc.) *causes* a change in rankings or traffic — only that certain signals are *associated with* pages that later showed movement.
- It does not claim to have proven anything about how Google's algorithm works. All of this is observational: we can say what was observed and what patterns look worth investigating further, not what caused what.

This keeps the framing as **decision support**: a way to help a person spend limited review time more wisely, not a claim of certainty about any individual page.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

url = "https://raw.githubusercontent.com/Houssem-Bjaoui/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Shape (lignes, colonnes):", df.shape)
print("\nColonnes disponibles:")
print(df.columns.tolist())

df.head()

Shape (lignes, colonnes): (30000, 44)

Colonnes disponibles:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [2]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  object 
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  object 
 7   main_intent             27626 non-null  object 
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   object 
 11  model_used              24267 non-null  object 
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d           30000 non-null

In [3]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
content_id,30000,30000,content_6880eb215048,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
client_id,30000,32,client_19581e27de,7008,NaN,NaN,NaN,NaN,NaN,NaN,NaN
search_volume,27532.0,NaN,NaN,NaN,158.882391,1518.270825,0.0,0.0,10.0,20.0,74000.0
competition,27532.0,NaN,NaN,NaN,0.146954,0.285241,0.0,0.0,0.0,0.13,1.0
competition_level,27390,3,LOW,22896,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cpc,27532.0,NaN,NaN,NaN,0.485342,2.10156,0.0,0.0,0.0,0.0,100.36
content_type,30000,3,keyword article,27207,NaN,NaN,NaN,NaN,NaN,NaN,NaN
main_intent,27626,4,informational,17235,NaN,NaN,NaN,NaN,NaN,NaN,NaN
word_count,22301.0,NaN,NaN,NaN,3107.760325,1452.382598,8.0,2413.0,2877.0,3666.0,9546.0
char_count,22301.0,NaN,NaN,NaN,20665.277835,10115.344042,40.0,15644.0,19116.0,24011.0,111158.0


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [8]:
# --- Number 1: overall trend_direction breakdown ---
trend_counts = df['trend_direction'].value_counts()
trend_pct = (df['trend_direction'].value_counts(normalize=True) * 100).round(1)
print("Trend direction counts:")
print(trend_counts)
print("\nTrend direction percentages:")
print(trend_pct)

down = df[df['trend_direction'] == 'down']

# --- Number 2: declining pages still on page 1 ---
page1_and_down = (down['position_tier'] == 'page_1').sum()
page1_and_down_pct = round(page1_and_down / len(down) * 100, 1)
print(f"\nDeclining pages: {len(down)}")
print(f"Declining AND still on page_1: {page1_and_down} ({page1_and_down_pct}%)")

# --- Number 3: declining pages with 'good' demand vs overall ---
good_impr_and_down = (down['impression_tier'] == 'good').sum()
good_impr_and_down_pct = round(good_impr_and_down / len(down) * 100, 1)
print(f"\nDeclining AND impression_tier == 'good': {good_impr_and_down} ({good_impr_and_down_pct}%)")
print("\nOverall impression_tier distribution (all pages):")
print((df['impression_tier'].value_counts(normalize=True) * 100).round(1))

# --- Supporting check: freshness among declining pages vs overall ---
print("\nfreshness_tier among declining pages:")
print((down['freshness_tier'].value_counts(normalize=True) * 100).round(1))
print("\nfreshness_tier overall:")
print((df['freshness_tier'].value_counts(normalize=True) * 100).round(1))

Trend direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Trend direction percentages:
trend_direction
down      54.2
stable    19.9
up        14.6
new        7.5
flat       3.8
Name: proportion, dtype: float64

Declining pages: 16262
Declining AND still on page_1: 6730 (41.4%)

Declining AND impression_tier == 'good': 4223 (26.0%)

Overall impression_tier distribution (all pages):
impression_tier
low          37.5
moderate     34.9
good         24.0
excellent     3.6
Name: proportion, dtype: float64

freshness_tier among declining pages:
freshness_tier
0-30      64.4
91-180    34.5
31-90      0.6
181+       0.5
Name: proportion, dtype: float64

freshness_tier overall:
freshness_tier
0-30      68.3
91-180    30.6
31-90      0.6
181+       0.6
Name: proportion, dtype: float64


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What the numbers above support saying:**

- More than half of pages in this dataset are **observed** to be in a declining trend (54.2%), which **suggests** review-time triage is genuinely needed, not a hypothetical problem.
- A meaningful share of declining pages (41.4%) **appear** to still hold page-1 positions, which **may** indicate an early-warning opportunity — a page worth reviewing before it potentially drops further.
- Declining pages **are associated with** "good" demand about as often as the dataset overall (26.0% vs 24.0%), which is **directional** evidence that decline is not confined to low-value pages, and is **worth investigating** further with a proper model.
- The freshness comparison (64.4% vs 68.3% in the most-recent tier) is a useful example of **not** overclaiming: the difference is small, so this notebook does **not** say "stale content causes decline" — the data does not support a claim that strong.

**What this work does *not* claim, and will not claim later:**

- It does **not** claim that `days_since_last_update`, `word_count`, or any other single feature *causes* a page's `trend_direction` to be "down." The relationships shown here are observed patterns in one dataset, not tested causal effects.
- It does **not** claim that refreshing a flagged page will definitely reverse the decline. Refreshing is an action a human may choose to take; whether it works would need to be tested afterward (e.g., by tracking pages post-refresh), which is outside the scope of this notebook.
- It does **not** claim to explain or predict how Google's ranking algorithm works. `avg_position` and `trend_direction` are outcomes observed in this dataset, not evidence about search engine mechanics.
- It does **not** claim that a future ML model will definitely solve this prioritization problem well. A model is a hypothesis to be tested — success will be judged later against a simple baseline rule, not assumed in advance.

In short: everything above describes **patterns observed in this specific dataset**, useful for deciding what is *worth investigating next* — not conclusions about causation, guaranteed outcomes, or how any search engine actually operates.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.